___
## a. The marginal probability of the first pixel
___

Let's discuss the conceptual approach to calculating the marginal probability of the first pixel, $P(x_1)$.

## Why is this considered easy?

Our model (PixelCNN) is built autoregressively. This means it always calculates a conditional probability: the probability of the current pixel, given all the pixels that preceded it: $P(x_i \mid x_{<i})$.

However, when dealing with the very first pixel in the image (the top-left corner at row 0, column 0), there is nothing preceding it. The set of "previous pixels" is simply an empty set.

Therefore, the conditional probability naturally (and mathematically) reduces to a pure marginal probability:


$$P(x_1 \mid \emptyset) = P(x_1)$$

## How does the architecture enforce this? (The Magic of Mask Type A)

Our approach relies on the network's structure at the code level. In the very first convolutional layer of PixelCNN, we use **Mask Type A**.

The role of this mask is to zero out the weight of the central pixel in the filter, along with all pixels to its right and directly below it.

The physical implication here is that during the forward pass of an image through the first layer, the neuron responsible for position $(0,0)$ is physically incapable of seeing the input value at $(0,0)$! It can only look "backward," and since there is no data there, it is fed exclusively by padding (zeros).

## The Solution Approach: The Dummy Image Trick

Because the neuron for the first pixel is completely "blind" to its own input, we arrive at a key insight: **it does not matter what image we feed into the model.**

We could input a real image of the digit 8 from MNIST, an image of random noise, or simply a black screen (an array of zeros)—the model's output at the specific $(0,0)$ position will be exactly the same.

If the input doesn't matter, where do the output values come from?

They come entirely from the **weights** and **biases** that the network learned during training. In other words, at this specific position, the network outputs its general (a priori) knowledge regarding which color is most likely to appear in the top-left corner of the dataset's images.

Therefore, the practical implementation is straightforward:

1. **Create an empty dummy image** filled entirely with zeros.
2. **Execute a single forward pass** by running this dummy image through the model.
3. **Extract the output vector (logits)** specifically at position $(0,0)$.
4. **Apply the Softmax function** to this vector.

The resulting output is the distribution across all 256 possible pixel intensities—which is exactly the learned marginal probability $P(x_1)$!

___
## b) The marginal probability of the middle pixel (pixel 14, 14).
___

Let's deeply analyze the theoretical and conceptual approach to Part B: calculating the marginal probability of the middle pixel, $P(x_{14,14})$.

In stark contrast to Part A (which was very easy), this task is classified as **HARD** (computationally intractable). Let's understand exactly why, and how a statistical approach allows us to bypass this limitation using a clever approximation.

## 1. Why does the direct approach fail? (The Mathematical Bottleneck)

Our model only knows how to calculate conditional probabilities: the probability of a pixel given the pixels that preceded it ($P(x_i \mid x_{<i})$).

The middle pixel of the image (at position 14,14) is located roughly at index 406 when the image is flattened (out of 784 total pixels). To find its marginal probability—meaning the likelihood of it taking a specific value without knowing anything else about the image—the laws of probability dictate we must perform **marginalization**. This means taking all possible combinations of the 405 preceding pixels, calculating the probability of each combination, and summing them up.

Mathematically, the equation looks like this:

$$P(x_{406}) = \sum_{x_1} \sum_{x_2} \dots \sum_{x_{405}} P(x_1, x_2, \dots, x_{406})$$

**Where is the problem?** Every preceding pixel can take 256 different values (from 0 to 255). The number of possible combinations we need to iterate over and sum is $256^{405}$. This is an astronomical number—vastly larger than the number of atoms in the observable universe. No computer in the world can perform this summation directly and precisely.

---

## 2. The Solution Approach: Monte Carlo Sampling Approximation

Since we cannot sum all possible combinations in existence, the research approach suggests the following: instead of going through *all* combinations, let's sample a representative subset of combinations (contexts / prefixes) and use them to estimate the average. This is the core principle of Monte Carlo approximation.

Instead of guessing completely random combinations (which would only confuse the model, as most would be pure white noise bearing no resemblance to an actual image), we use the model itself to generate these contexts!

### How does this approach work step-by-step?

1. **Generating Contexts:** We run the model in a standard autoregressive manner (pixel by pixel) and generate $N$ different images (e.g., $N=32$ or $N=64$).
2. **Stopping in Time:** We halt the generation process exactly one pixel before the middle pixel (i.e., we only generate the first 405 pixels). We now have $N$ valid, logical "half-images" that the model produced.
3. **Querying the Model:** For each of these generated contexts, we feed it back into the model and ask: *"Given this specific generated half-image, what is the probability distribution of the next pixel (the middle one)?"* The model returns a vector of 256 probabilities for each context.
4. **Averaging:** According to the Law of Large Numbers, if we take these $N$ probability vectors and calculate a simple arithmetic mean, we will obtain an excellent approximation of the true marginal probability!

The equation for this approach looks like this:

$$P(x_{14,14}) \approx \frac{1}{N} \sum_{i=1}^N P(x_{14,14} \mid x_{<14,14}^{(i)})$$

### When is this approximation good, and when is it problematic?

* **Why is it excellent for this task?** It solves the problem with very low computational complexity. Instead of infinite computation time, we run the generation loop only 405 times for a batch of $N$ samples in parallel, yielding a result in mere seconds.
* **What is the downside?** If $N$ is too small (e.g., sampling only 2-3 images), the approximation will be poor and overly dependent on the "luck" of the draw. As we increase $N$ (e.g., to 64 or 128), the approximation becomes much more stable and accurate, though the runtime will slightly increase.

This is the theoretical foundation behind the approximation code we will write next.

Here is the exact breakdown of how the steps work in practice:

**1. Generating the Full Distribution:**
For half-image #1, the model outputs a vector of 256 probabilities (for example: a 10% chance the pixel intensity is 0, a 50% chance it is 128, etc.).

**2. Collection:**
We generate and collect $N$ such vectors (each of length 256), one for every half-image (context) we produced.

**3. Element-wise Averaging:**
We calculate the average probability for color intensity 0 across all $N$ images. Then, we average the probability for color intensity 1 across all $N$ images, and so on.

**4. The Result (The Marginal Approximation):**
We obtain a single, new vector of length 256. This vector represents our approximated marginal probability distribution for the middle pixel, independent of any specific context.

---

## When *do* we look for the maximum?

Only at the very end of the process.

After we have calculated the final averaged vector (which contains 256 values), we can examine it and conclude: *"Based on the marginal distribution we computed, the color intensity with the highest probability (the $\arg\max$) is $X$."*

If we had only looked at the maximum value within each individual half-image and averaged those, we would have converted "soft" probabilities into hard labels. Doing so would throw away a massive amount of valuable statistical information that is absolutely vital for obtaining an accurate approximation using the Law of Large Numbers.

___


There are three main reasons (both mathematical and practical) that make Monte Carlo the standard method in both industry and academia for solving such problems:

### 1. There is no other choice (Intractability)

In the ideal world of mathematics, we would compute the exact result using marginalization (summing over all possible contexts). But as we have seen, the space here is of size $256^{405}$.

When a problem is intractable (computationally infeasible in a reasonable timeframe), the scientific community accepts probabilistic approximations as the only realistic option. Monte Carlo turns a computational problem that would take longer than the age of the universe into one that can be solved in 10 seconds on Colab.

### 2. The Mathematical Foundation: The Law of Large Numbers

Monte Carlo is not just a "shot in the dark." It rests on a solid mathematical theorem.

Statistically, the marginal probability we are looking for is equal to the expected value (Expectation) of the conditional probabilities, under the data distribution of the images. That is:

$$P(x_{\text{mid}}) = \mathbb{E}_{\text{context} \sim P}[P(x_{\text{mid}} \mid \text{context})]$$

The Law of Large Numbers states that if it is impossible to calculate the expected value over the entire population of the universe (all possible images), we can take a sample of $N$ observations and calculate their mean. The sample mean is an unbiased estimator of the true expected value. As $N$ (the number of half-images we generate) grows larger, the approximation will converge and approach the true mathematical value with certainty.

### 3. We are sampling from the correct distribution

This is perhaps the most important part to understand why this approximation is good: we are not drawing "random noise" and using it as context.

If we were to insert uniform random noise into the first 405 pixels, Monte Carlo would give us a terrible result, because we would be querying the model on illogical images it has never seen.

But in our case, we are using the PixelCNN model itself to generate the half-images (using a method called **Ancestral Sampling**).

This means our samples come from the exact same distribution as the dataset the model was trained on (for example, MNIST digits). Therefore, the $N$ samples we draw are highly meaningful representations of reality, which makes the mean a highly accurate approximation even with a relatively small number of samples (like 32 or 64).

___
The equation you provided is:


$$P(x_{\text{mid}}) = \mathbb{E}_{\text{context} \sim P}[P(x_{\text{mid}} \mid \text{context})]$$

To understand it deeply, let's break it down to the fundamental mathematical definition of expected value, and see how it connects to the Law of Total Probability.

## 1. The General Definition of Expectation for a Discrete Variable

For any discrete random variable $Y$, the expectation (expected value) is defined as the sum of the products of every possible value the variable can take, multiplied by the probability of taking that value:


$$\mathbb{E}[Y] = \sum_{y} y \cdot P(y)$$

## 2. Translating the Definition to Our Problem (PixelCNN)

In our problem, the "sample space" (all possible states) is the set of all possible contexts. Let's denote a specific context (a half-image) with the letter $c$.

The variable for which we want to calculate the expected value is the conditional function: $P(x_{\text{mid}} \mid c)$.

The probability that this context $c$ occurs in reality (the probability of generating this specific half-image) is $P(c)$.

Therefore, if we plug this into the definition of the expectation for a discrete variable, we get exactly the **Law of Total Probability**:


$$P(x_{\text{mid}}) = \mathbb{E}_{c \sim P}[P(x_{\text{mid}} \mid c)] = \sum_{c \in \text{All Contexts}} P(x_{\text{mid}} \mid c) \cdot P(c)$$

**What does this equation mean in plain English?**
The marginal probability of the middle pixel is equal to the sum over all possible half-images in the universe. For each half-image, we take the probability of the middle pixel given that specific half-image, and multiply it by its weight—which is the probability that this half-image would appear in the first place.

## 3. The Connection to Monte Carlo

As we discussed earlier, the set of "All Contexts" in our case includes $256^{405}$ possibilities, which makes it impossible to compute this sum in its entirety.

This is where the mathematical justification for **Monte Carlo sampling** comes in. Instead of summing over the entire massive sample space and multiplying by $P(c)$, we randomly sample $N$ contexts from the distribution $P(c)$ (i.e., we generate half-images directly from the model).

When we sample from the true distribution, the weight $P(c)$ is inherently factored into the sampling process itself (common contexts will be sampled more frequently, rare contexts less so). Therefore, the complex mathematical sum simplifies into a straightforward average over our sample:


$$\sum_{c} P(x_{\text{mid}} \mid c) \cdot P(c) \approx \frac{1}{N} \sum_{i=1}^{N} P(x_{\text{mid}} \mid c^{(i)})$$

Where $c^{(i)}$ are those exact half-images we generated in the code loop. The expected value of the probabilities in our sample converges to the true expected value as $N$ grows larger.


https://en.wikipedia.org/wiki/Law_of_large_numbers

___
## c) The conditional probability of the middle pixel, given the values of all pixels above and to the left of it
___

Let's dive into the theoretical depth of Part C. Of all the sections in this assignment, this part represents the purest essence of what PixelCNN is designed to do. This is the model's "comfort zone," and on an academic level, solving this section demonstrates a true understanding of autoregressive network architectures.

## The Perfect Mathematical Fit (Native Autoregression)

The question asks for:

$$P(x_{14,14} \mid x_{<14,14})$$

That is, the probability of the middle pixel, given all the pixels located above it and to its left.

The PixelCNN network is mathematically defined by the Chain Rule of Probability:

$$P(X) = \prod_{i=1}^{N} P(x_i \mid x_{<i})$$

If we flatten our image into a sequence of pixels from left to right and top to bottom, the central pixel at $(14,14)$ is located exactly in the middle of the sequence. The set of pixels $x_{<14,14}$ consists precisely of all the pixels that preceded it in this scan (all the rows above it, and the pixels to its left in the same row).

Therefore, unlike other sections that required us to perform mathematical approximations or integrate over missing variables (Marginalization), here the question asks us to compute a function that the network models directly and naturally.

## The Architectural Mechanism: How Does the Network Enforce This Condition?

How can we be sure that the network's output at position $(14,14)$ was truly influenced *only* by the pixels above and to its left, without accidentally "peeking" at future pixels?

The answer lies in **Masked Convolutions**:

* **Restricting the Receptive Field:** The convolutions in the network (using Mask Type A in the first layer and Mask Type B in subsequent layers) are constructed to physically zero out the weights of anything located to the right and below the central pixel of the filter.

This means that when the network performs a computation for the coordinate $(14,14)$, all the information flowing into this neuron comes exclusively from the subset of permitted pixels (its history). The network doesn't need us to "hide" the rest of the image from it—it is simply blind to it at the hardware and tensor level.

## The Solution Process (Conceptual Approach)

To actually obtain this probability in practice, the process is entirely straightforward and requires no Monte Carlo sampling or loops:

1. **Providing the Context (The "Given" Condition):** Since we want to calculate a conditional probability, we need to provide the model with the condition—namely, the actual pixels that came before. To do this, we grab a single, valid, real image from the Validation or Test set.
2. **Forward Pass:** We feed this complete image into the model exactly as is. Even though we input the entire image (including the "future" pixels), because of the masks discussed above, the computation at the middle position will not be influenced by the future in any way.
3. **Isolating the Relevant Output:** The model will output a massive tensor containing distributions for all 784 pixels in the image. We simply slice out the specific vector located at coordinates $(14,14)$.
4. **Converting Logits to Probabilities:** The extracted vector will contain 256 real numbers (logits). To convert them into valid probabilities (all between 0 and 1, where the sum equals 1), we pass the vector through a Softmax function.

The vector outputted by the Softmax function is the final answer to this section. It accurately represents the model's percentage distribution for each of the 256 possible shades, based solely on what it observed in the first half of the real image we fed into it.

___
## d) The conditional probability of the middle pixel, given the values of all pixels below and to the right of it.
___


**Classification: HARD**

**1. Architectural Constraint (Causal Blindness)**
Autoregressive models like PixelCNN are strictly unidirectional by design. The network models the joint distribution of an image by factorizing it into a product of conditional probabilities: $P(x_i \mid x_{<i})$. Due to the causal masking mechanism (Mask Types A and B), the network's receptive field for a given target pixel is strictly restricted to the pixels above and to the left of it. Consequently, the model is architecturally "blind" to any pixels located below or to the right of the target. It cannot natively evaluate $P(x_{14,14} \mid x_{>14,14})$.

**2. Computational Intractability of Exact Inference**
To calculate this probability exactly, we would need to use Bayes' Theorem and marginalize over all possible states of the unobserved "past" pixels ($x_{<14,14}$):
$$P(x_{mid}=c \mid x_{future}) \propto \sum_{x_{past}} P(x_{mid}=c, x_{future} \mid x_{past}) \cdot P(x_{past})$$
Summing over $256^{405}$ past combinations is computationally intractable. 

**3. Approximation Method (Monte Carlo + Likelihood Evaluation)**
Since exact inference is too costly, we approximate the distribution using a single sampled context:

* **Step 1 (Sample a Past):** Use the model autoregressively to generate a single, highly-probable valid "past" context ($x_{past}^*$).
* **Step 2 (Hypothesis Generation):** Combine this sampled past with the *given* real future context to create a batch of 256 composite images. Inject a unique intensity value $c \in [0, 255]$ into the middle pixel of each image across the batch, creating 256 parallel hypotheses.
* **Step 3 (Likelihood Evaluation):** Pass the entire batch of 256 hypotheses through the model simultaneously. For each color hypothesis $c$, calculate the sum of the Cross-Entropy loss (negative log-likelihood) strictly over the middle pixel and all future pixels:
$$L_c = - \sum_{i \ge mid} \log P(x_i \mid x_{<i}^{(c)})$$
Where $x_{<i}^{(c)}$ is the specific causal context for pixel $i$ under hypothesis $c$. This loss $L_c$ acts as a measure of "surprise"—quantifying how statistically unlikely the future sequence is, given the specific color $c$ injected into the middle pixel.
* **Step 4 (Normalization):** To translate these 256 loss values ($L_0 \dots L_{255}$) back into a valid probability distribution, we negate them to restore the log-likelihoods and apply the Softmax function. This exponentiates and normalizes the scores so they sum exactly to 1.0:
$$P(x_{mid}=c \mid x_{future}) \approx \text{Softmax}(-L_c) = \frac{\exp(-L_c)}{\sum_{v=0}^{255} \exp(-L_v)}$$

___
## e) The conditional probability of the middle pixel, given the values of all other pixels in the image
___

### 3(e) The conditional probability of the middle pixel, given the values of all other pixels in the image.

**Classification: EASY (Exactly Computable)**

**1. The Mathematical Foundation (Bayes' Theorem)**
We are asked to calculate $P(x_{14,14} \mid x_{\neq 14,14})$. In part (d), missing the entire "past" created an intractable state space of $256^{405}$. Here, however, we are given *all* other pixels. The only unknown is a single discrete variable (the middle pixel) which can only take 256 possible states. 

Using the definition of conditional probability, we can express this exact distribution as the ratio of the joint probability of the full image to the sum of all possible full images:
$$P(x_{mid} = c \mid x_{\neq mid}) = \frac{P(\text{Full Image with } x_{mid}=c)}{\sum_{v=0}^{255} P(\text{Full Image with } x_{mid}=v)}$$

**2. The Autoregressive "Ripple Effect"**
How does the network condition on the future here? It happens naturally through the joint probability factorization:
$$P(X) = P(x_{<mid}) \cdot P(x_{mid} \mid x_{<mid}) \cdot P(x_{>mid} \mid x_{\le mid})$$
If we change the color of the middle pixel, it directly alters the conditional probability of **all subsequent future pixels** ($x_{>mid}$). For example, if we inject a bright white pixel into a dark background, the network will assign a severe penalty (low probability) to the future dark pixels because they logically shouldn't follow a bright white pixel. By calculating the log-likelihood of the *entire* image, the future pixels essentially cast a "vote" on whether the middle pixel's color makes sense.

**3. Algorithmic Steps**
To compute this efficiently:

* **Step 1 (Batching):** Take the real given image and clone it 256 times.
* **Step 2 (Hypothesis Injection):** Systematically replace the middle pixel in each clone with a unique intensity value $c \in [0, 255]$. We now have 256 complete, fully-defined images representing every possible reality.
* **Step 3 (Total Likelihood):** Pass the batch of 256 hypotheses through the model. Calculate the total Cross-Entropy loss (negative log-likelihood) for the *entire* image (all $N$ pixels) for each hypothesis $c$:
$$L_c = - \sum_{i=1}^{N} \log P(x_i \mid x_{<i}^{(c)})$$
* **Step 4 (Normalization):** Negate the total image losses to restore the log-likelihoods, and apply a Softmax function. This exponentiates the values and normalizes them, yielding the exact, true conditional probability distribution:
$$P(x_{mid}=c \mid x_{\neq mid}) = \text{Softmax}(-L_c) = \frac{\exp(-L_c)}{\sum_{v=0}^{255} \exp(-L_v)}$$

**4. Mathematical Proof: The Softmax Constant Cancellation**
In Step 3, we calculate the loss over the *entire* image, instead of applying a mask to ignore the "past" pixels ($x_{<mid}$). We can prove that this is mathematically identical to applying a mask. 

Let's split the total negative log-likelihood $L_c$ into the sum of the "past" pixels and the "rest" (middle + future) of the image. Mathematically, these are defined as:
$$L_{past} = - \sum_{i < mid} \log P(x_i \mid x_{<i})$$
$$L_{rest}^{(c)} = - \sum_{i \ge mid} \log P(x_i \mid x_{<i}^{(c)})$$

*(Note that $L_{past}$ does not have a $(c)$ superscript because the predictions for the past pixels are not affected by the injected middle pixel).*

Therefore, the total loss is:
$$L_c = L_{past} + L_{rest}^{(c)}$$

Since the past pixels are identical across all 256 hypotheses, their loss $L_{past}$ is a constant scalar, which we will call $K$. Thus, $L_c = K + L_{rest}^{(c)}$.

When we plug this into the Softmax equation from Step 4:
$$P(x_{mid}=c \mid x_{\neq mid}) = \frac{\exp(-(K + L_{rest}^{(c)}))}{\sum_{v=0}^{255} \exp(-(K + L_{rest}^{(v)}))}$$

Using the laws of exponents ($\exp(A+B) = \exp(A) \cdot \exp(B)$), we can separate the constant $K$:
$$P(x_{mid}=c \mid x_{\neq mid}) = \frac{\exp(-K) \cdot \exp(-L_{rest}^{(c)})}{\sum_{v=0}^{255} [\exp(-K) \cdot \exp(-L_{rest}^{(v)})]}$$

Because $\exp(-K)$ is independent of the summation index $v$, we can factor it completely out of the sum in the denominator:
$$P(x_{mid}=c \mid x_{\neq mid}) = \frac{\exp(-K) \cdot \exp(-L_{rest}^{(c)})}{\exp(-K) \cdot \sum_{v=0}^{255} \exp(-L_{rest}^{(v)})}$$

The $\exp(-K)$ terms cancel out perfectly from both the numerator and the denominator:
$$P(x_{mid}=c \mid x_{\neq mid}) = \frac{\exp(-L_{rest}^{(c)})}{\sum_{v=0}^{255} \exp(-L_{rest}^{(v)})}$$

**Conclusion:** Computing the loss over the entire image implicitly and perfectly ignores the identical past pixels. The shared "past" simply scales the numerator and denominator equally, resulting in the exact same final probability distribution without requiring any explicit tensor masking in the code.

---

**A Concrete 5-Pixel Example:**
To visualize exactly why $L_{past}$ is a constant $K$ and $L_{rest}$ changes, imagine a simplified 1D image with only 5 pixels: $(x_1, x_2, x_3, x_4, x_5)$. 
We want to find the conditional probability of the middle pixel ($x_3$). We inject our color hypothesis $c$, making our test sequence: $(x_1, x_2, c, x_4, x_5)$.

By the chain rule, the total negative log-likelihood $L_c$ is:
$$L_c = \underbrace{-\log P(x_1) - \log P(x_2 \mid x_1)}_{L_{past}} \underbrace{- \log P(x_3=c \mid x_1, x_2) - \log P(x_4 \mid x_1, x_2, c) - \log P(x_5 \mid x_1, x_2, c, x_4)}_{L_{rest}^{(c)}}$$

Breaking it down:
* **The Past ($L_{past}$):** Notice that $c$ does not appear anywhere in the conditioning of $x_1$ and $x_2$. No matter which of the 256 colors we inject into $x_3$, the loss for the first two pixels remains exactly the same. This is our constant $K$.
* **The Rest ($L_{rest}^{(c)}$):** Here, $c$ affects its own probability at position 3, and then "ripples" forward into the given context of $x_4$ and $x_5$. If $c$ is a bad fit for the surrounding pixels, these future terms will output a high loss. This is the dynamic part that differentiates our 256 hypotheses.

 so this is also the last step we done in part d? 



## The "Aha!" Moment: Connecting Part D and Part E

Excellent question, and you've hit on a brilliant point!

The short answer is: **Yes, absolutely!** This mathematical logic applies 100% to Part D as well. Let's connect this directly to what we did, because it shows you've understood the mechanism on a very deep level:

In Part D (the probability given the "future"), although we *sampled* the "past" (instead of taking a real past like in Part E), once we sampled it in Step 1—we copied that exact same past to all 256 images.

Therefore, in Part D as well, the loss function of the top half of the image ($L_{\text{past}}$) was a constant ($K$) across all 256 hypotheses in the batch.

**So what is the difference between our code in Part D and our code in Part E?**

Here comes the real "click" of understanding:

* **In Part D (What we actually did):** We did it the "safe" and **explicit** way. We built a binary mask of zeros and ones that physically zeroed out the loss of the "past" before we called `sum`. In other words, instead of relying on the Softmax cancellation, we proactively stripped $K$ out using tensors, computing only $L_{\text{rest}}$.
* **In Part E (What we just proved):** We did it the "elegant" and **implicit** mathematical way. We threw the mask in the trash, summed the loss over the *entire* image (meaning we computed $K + L_{\text{rest}}$), and let the mathematical shift-invariance of the Softmax function cancel out $K$ entirely on its own.

### The Conclusion (and the Bonus!)

Thanks to this mathematical proof, you now realize that if you went back to your code for Part D, completely deleted the lines that create the mask (and calculated the sum over the entire image instead)—**you would get exactly, but *exactly*, the same result!**

Part E simply gave us the opportunity to showcase this mathematical trick, which saves us the hassle of building cumbersome masks whenever we have identical image segments across our batch. Excellent observation on your part!

- - - 
## f) Samples of the bottom half of an image, given the top half of an image.
- - -

### 3(f) Samples of the bottom half of an image, given the top half of an image.

**Classification: EASY (Natively Supported via Ancestral Sampling)**

**1. Architectural Alignment (Forward Time)**
Unlike part (d) which asked the model to predict the past given the future, this task asks the model to predict the future given the past. This perfectly aligns with the causal ordering of PixelCNN. The pixels in the top half (rows 0 to 13) are exactly the set $x_{<i}$ for any pixel $i$ located in the bottom half (rows 14 to 27). Therefore, the model's masked convolutions naturally support this conditioning without requiring any approximations, Bayes' theorem inversions, or masked loss calculations.

**2. The Mathematical Foundation (Ancestral Sampling)**
To generate the bottom half, we are sampling from the joint conditional distribution:
$$P(X_{bottom} \mid X_{top}) = \prod_{i \in bottom} P(x_i \mid x_{<i})$$
Because this distribution is factorized autoregressively, we cannot sample the entire bottom half at once. Instead, we must use **Ancestral Sampling** (also known as sequential sampling). We iterate through the missing pixels one by one in raster-scan order. For each pixel, we compute its exact probability distribution given the top half *and* the previously sampled pixels of the bottom half, draw a random value from that distribution, and feed it back into the image before moving to the next pixel.

**3. Algorithmic Steps**
* **Step 1 (Conditioning):** Extract the top half (rows 0-13) from a real dataset image.
* **Step 2 (Batching for Diversity):** Since the prompt asks for "samples" (plural), we copy this exact same top half into a batch of $B$ blank images. This will allow us to generate multiple different, statistically valid bottom halves for the exact same top half, proving the model is stochastic.
* **Step 3 (Sequential Generation):** Loop through rows 14 to 27, and columns 0 to 27. At each coordinate $(h,w)$:
  * Pass the batch through the model.
  * Extract the 256-dimensional Softmax probability vector at $(h,w)$.
  * Sample a pixel intensity $c \sim P(x_{h,w} \mid x_{<h,w})$ using `torch.multinomial`.
  * Inject the sampled values into the batch and proceed to the next coordinate.

- - - 
## g) Samples of the top half of an image, given the bottom half of an image.
- - - 

**Classification: HARD**

**1. The Symmetrical Problem (Predicting the Past)**
Unlike part (f), this task asks us to sample the "past" (top half, $x_{<14}$) conditioned entirely on the "future" (bottom half, $x_{\ge 14}$). Because PixelCNN is an autoregressive causal model, it can only naturally sample forwards. Natively sampling the top half conditioned on the bottom half is architecturally impossible in a single forward pass.

**2. The Mathematical Foundation (Bayes' Theorem & Importance Sampling)**
We want to sample from the posterior distribution: $P(X_{top} \mid X_{bottom})$.
Using Bayes' theorem, this is proportional to:
$$P(X_{top} \mid X_{bottom}) \propto P(X_{bottom} \mid X_{top}) \cdot P(X_{top})$$

To approximate sampling from this intractable distribution, we can use a method inspired by **Sampling Importance Resampling (SIR)**:
1. **$P(X_{top})$ [The Prior]:** We can natively sample valid top halves using standard ancestral sampling, totally blind to the bottom half.
2. **$P(X_{bottom} \mid X_{top})$ [The Likelihood]:** Once we have a proposed top half, we can attach the known bottom half and evaluate how "likely" this specific future is, given our proposed past.

**3. Algorithmic Steps**
* **Step 1 (Generate Proposals):** Unconditionally sample a large batch (e.g., $N=64$) of valid top halves using the model's standard autoregressive generation. These are our hypotheses.
* **Step 2 (Attach Future):** Take the real given bottom half and append it to all 64 generated top halves to create 64 complete composite images.
* **Step 3 (Likelihood Evaluation):** Pass the entire batch through the model. Calculate the Cross-Entropy loss (NLL) *strictly* over the bottom half. This loss acts as a penalty: a top half that logically flows into the real bottom half will yield a low loss.
* **Step 4 (Resampling):** Negate the losses and apply Softmax to turn them into importance weights (probabilities). Sample from these 64 composite images based on these weights to select the most statistically sound top halves.